# Phase 2i: IQtree Building

Generates distance tree using IQTree.

<details>
    <summary>Click To See A Decription of Parameters</summary>
        <pre>
            <code>
save_dir: str
    Path to directory for saving outputs in.

fasta_path: str
    Name used for fasta files containing sequences.

max_threads: int, default None
    The maximum number of threads to use when calling gnu parallel.

tree_dir_name: str
    Name of directory to place outputs of IQtree in.

seed: int
    Seed to use when calling IQtree.

  </code>
</pre>

In [ ]:
save_dir=''
fasta_path=''
max_threads=''
tree_dir_name=''
kernel_name='beast_pype'
seed=''

In [ ]:
if [ "$seed" = "" ];
then
    seed=''
else
    seed="--seed $seed"
fi

In [ ]:
# For some reason this still has to be called so that slurm's sbatch to use the beast_pype environment.
source activate $kernel_name

In [ ]:
default_threads=11
if [ $max_threads -gt $default_threads ]; then
    cores_to_use=$default_threads
    jobs_per_time=$((max_threads / default_threads))
else
    cores_to_use=$max_threads
    jobs_per_time=1
fi

In [ ]:
if [ -f "${fasta_path}" ]; then
    mkdir ${save_dir}/${tree_dir_name}
    iqtree -ninit 2 -n 2 -me 0.05 -nt $cores_to_use -s ${fasta_path} -m GTR -ninit 10 -n 4\
            $seed\
            --prefix ${save_dir}/${tree_dir_name}/iqtree > ${save_dir}/${tree_dir_name}/iqtree.out
    mv ${save_dir}/${tree_dir_name}/iqtree.treefile ${save_dir}/${tree_dir_name}/iqtree.nwk
else
    valid_dirs=$(find ${save_dir} -name ${fasta_path} -printf '%h\n' | sort -u)
    for directory in ${valid_dirs[@]}; do mkdir ${directory}/${tree_dir_name} ; done
    parallel --jobs=$jobs_per_time --results {1}/${tree_dir_name}/iqtree_\
        iqtree -ninit 2 -n 2 -me 0.05 -nt $cores_to_use -s {1}/${fasta_path} -m GTR -ninit 10 -n 4\
        $seed\
        --prefix {1}/${tree_dir_name}/iqtree\
        ::: ${valid_dirs[@]}
    for directory in ${valid_dirs[@]}
        do
        mv ${directory}/${tree_dir_name}/iqtree.treefile ${directory}/${tree_dir_name}/iqtree.nwk
        done
fi